<a href="https://colab.research.google.com/github/fernandinhaalmeida84/Tech_Challenge_Fase_2/blob/main/Notebook/03_Modelagem_Preditiva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 03: Modelagem Preditiva**

Neste notebook, realizaremos o treinamento, otimização e avaliação dos modelos de classificação de risco de crédito utilizando a base previamente tratada e consolidada.

In [16]:
# 1. Baixa o arquivo zip direto do GitHub (usando ?raw=true) para o ambiente do Colab
!wget -O /content/BaseDadosFase2.zip https://github.com/fernandinhaalmeida84/Tech_Challenge_Fase_2/blob/1462573e4453793ed0c8e218b35ece6ce166caee/Data/Raw/BaseDadosFase2.zip?raw=true

# 2. Descompacta o arquivo em uma pasta chamada 'dados_extraidos'
!unzip -o /content/BaseDadosFase2.zip -d dados_extraidos

--2026-10-08 23:39:42--  https://github.com/fernandinhaalmeida84/Tech_Challenge_Fase_2/blob/1462573e4453793ed0c8e218b35ece6ce166caee/Data/Raw/BaseDadosFase2.zip?raw=true
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/fernandinhaalmeida84/Tech_Challenge_Fase_2/raw/1462573e4453793ed0c8e218b35ece6ce166caee/Data/Raw/BaseDadosFase2.zip [following]
--2026-10-08 23:39:42--  https://github.com/fernandinhaalmeida84/Tech_Challenge_Fase_2/raw/1462573e4453793ed0c8e218b35ece6ce166caee/Data/Raw/BaseDadosFase2.zip
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/fernandinhaalmeida84/Tech_Challenge_Fase_2/1462573e4453793ed0c8e218b35ece6ce166caee/Data/Raw/BaseDadosFase2.zip [following]
--2026-10-08 23:39:42--  https://raw.githubusercontent.com/fernandinhaalmeida8

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

# 1. Definição de caminhos e constantes
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', None)

PATH_BASE = Path('/content/Data/Processed/base_modelagem.parquet')

# 2. Carregando a base de modelagem
if PATH_BASE.exists():
    df_modelagem = pd.read_parquet(PATH_BASE)
    print(f"✅ Base de modelagem carregada com sucesso! Dimensões: {df_modelagem.shape}")
    display(df_modelagem.head(3))
else:
    print("❌ Arquivo 'base_modelagem.parquet' não encontrado. Certifique-se de executar as etapas anteriores primeiro.")

🔄 Carregando dados brutos e construindo a base de modelagem...
✅ Base de modelagem consolidada com sucesso! Dimensões: (36457, 21)


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET,MESES_HISTORICO,PERFIL_ID
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,1,16,M_427500.0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,1,15,M_427500.0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,0,30,M_112500.0


## **1. Engenharia de atributos**

Transforma os dados originais em variáveis mais relevantes para o modelo de Machine Learning, de forma a facilitar a identificação de padrões e melhorar a capacidade de prever a inadimplência dos clientes.


In [18]:
df['IDADE'] = (-df['DAYS_BIRTH'] / 365.25).round(1)
# DAYS_EMPLOYED positivo (365243) indica aposentado/sem vínculo
df['SEM_EMPREGO_ATUAL'] = (df['DAYS_EMPLOYED'] > 0).astype(int)
df['ANOS_EMPREGO'] = np.where(df['DAYS_EMPLOYED'] > 0, 0, -df['DAYS_EMPLOYED'] / 365.25)
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Nao_informado')
df['RENDA_POR_MEMBRO'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS'].clip(lower=1)

# Fora do modelo: identificadores, alvo, informação pós-aprovação e coluna constante
DESCARTAR = ['ID', 'TARGET', 'TARGET_30', 'MESES_HISTORICO', 'PERFIL_ID',
             'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL']
X = df.drop(columns=[c for c in DESCARTAR if c in df.columns])
y = df['TARGET']
grupos = df['PERFIL_ID']

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()
print('Categóricas:', cat_cols)


Categóricas: ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']


##**2. Separar treino e teste por grupo de perfil (para evitar vazamento)**

In [19]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
idx_tr, idx_te = next(sgkf.split(X, y, grupos))   # ~80% treino / ~20% teste

X_train, X_test = X.iloc[idx_tr], X.iloc[idx_te]
y_train, y_test = y.iloc[idx_tr], y.iloc[idx_te]
g_train = grupos.iloc[idx_tr]

assert set(grupos.iloc[idx_tr]).isdisjoint(grupos.iloc[idx_te]), 'Perfil vazou!'
print(f"Treino: {X_train.shape} | taxa mau = {y_train.mean():.4f}")
print(f"Teste : {X_test.shape} | taxa mau = {y_test.mean():.4f}")


Treino: (30246, 18) | taxa mau = 0.1157
Teste : (6211, 18) | taxa mau = 0.1274


**Análise**

* Treino: contem 30.246 amostras e 18 variáveis explicativas.

* Teste: contém 6.211 amostras e 18 variáveis.

Do total de 36.457 registros, 82,9% foram direcionados para treino e 17% ao teste. Proporção esta condizentes coma divisão de 5 folds (80%/20%).

Já a proporção de Maus_pagadores:
* Treino 11,5%
* Teste: 12,7%
Amostragem equilibrada.

## **3. Pipeline de pré-processamento e modelos**

O baseline (classe majoritária) mostra que acurácia alta sozinha não significa modelo útil. O class_weight='balanced' trata o desbalanceamento sem criar dados artificiais.

In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

prep = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)])

modelos = {
    'Baseline (classe majoritária)': DummyClassifier(strategy='most_frequent'),
    'Regressão Logística': LogisticRegression(max_iter=2000, class_weight='balanced',
                                              random_state=RANDOM_STATE),
    'Árvore de Decisão': DecisionTreeClassifier(max_depth=6, class_weight='balanced',
                                                random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            class_weight='balanced', n_jobs=-1,
                                            random_state=RANDOM_STATE),
    'HistGradientBoosting': HistGradientBoostingClassifier(class_weight='balanced',
                                                           random_state=RANDOM_STATE),


## **4. Validação cruzada por grupo e métricas no teste**

In [23]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
resultados, pipes = [], {}

for nome, modelo in modelos.items():
    pipe = Pipeline([('prep', prep), ('modelo', modelo)])
    cvres = cross_validate(pipe, X_train, y_train, groups=g_train, cv=cv,
                           scoring=['average_precision', 'roc_auc'], n_jobs=-1)
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    resultados.append({
        'Modelo': nome,
        'AUC-PR (CV)': cvres['test_average_precision'].mean(),
        'AUC-ROC (CV)': cvres['test_roc_auc'].mean(),
        'Acurácia': accuracy_score(y_test, pred),
        'Precisão': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred, zero_division=0),
        'AUC-ROC': roc_auc_score(y_test, proba),
        'AUC-PR': average_precision_score(y_test, proba)})
    pipes[nome] = pipe

tabela = pd.DataFrame(resultados).set_index('Modelo').round(3)
display(tabela.sort_values('AUC-PR (CV)', ascending=False))


,AUC-PR (CV),AUC-ROC (CV),Acurácia,Precisão,Recall,F1,AUC-ROC,AUC-PR
Modelo,,,,,,,,
Regressão Logística,0.141,0.531,0.507,0.121,0.458,0.191,0.492,0.142
Árvore de Decisão,0.125,0.516,0.439,0.130,0.597,0.213,0.510,0.134
HistGradientBoosting,0.121,0.499,0.666,0.118,0.252,0.161,0.496,0.124
Random Forest,0.118,0.510,0.839,0.036,0.010,0.016,0.508,0.127
Baseline (classe majoritária),0.116,0.500,0.873,0.000,0.000,0.000,0.500,0.127


## **Análise Comparativa dos Modelos**

A validação dos modelos foi conduzida utilizando Validação Cruzada com Estratificação e Agrupamento (StratifiedGroupKFold) no conjunto de treino e confirmada no conjunto de Teste (Holdout). Para contornar a ilusão estatística causada pelo desbalanceamento da classe de inadimplentes (~12% de maus pagadores), a métrica primária adotada para seleção e ordenação do modelo foi o AUC-PR (Validação Cruzada).

###**Principais Achados**

•	Melhor Desempenho Relativo: A Regressão Logística apresentou o melhor resultado geral, liderando no AUC-PR (CV = 0,141 / Teste = 0,142) e alcançando um Recall de 45,8% no limiar padrão (0,5).

•	Evidência da Armadilha da Acurácia: O modelo Baseline obteve a maior Acurácia (87,3%), porém com Recall = 0,000, falhando totalmente na detecção de maus pagadores. O Random Forest apresentou padrão similar: acurácia elevada (83,9%), mas identificando apenas 1,0% dos inadimplentes (Recall = 0,010).

•	Baixo Poder Discriminatório da Base: Todos os modelos registraram métricas de AUC-ROC próximas de 0,50 (faixa de 0,499 a 0,531 em CV). Considerando que o baseline aleatório para o AUC-PR nesta taxa de evento é de ~0,116, os ganhos obtidos mesmo pelos algoritmos mais complexos (Ensembles) foram mínimos.

###**Conclusão Técnica e Próximos Passos**

1.	Diagnóstico do Problema: O desempenho próximo do aleatório em todos os algoritmos indica que o fator limitante não é a escolha da técnica de Machine Learning, mas sim o baixo poder preditivo do conjunto de atributos (features) atual.

2.	Engenharia de Atributos (Feature Engineering): É indispensável criar novas variáveis de negócio (razões financeiras, métricas comportamentais ou contagens temporais) para gerar sinal preditivo real.

3.	Ajuste do Limiar de Decisão (Threshold Tuning): Para modelos como a Regressão Logística ou Árvore de Decisão, ajustar o ponto de corte da probabilidade prevista (abaixo dos 0,5 padrão) permitirá otimizar o balanço financeiro entre falsos positivos e falsos negativos.
